# Viveka — Llama-3.2-3B Inference (fills the capacity gradient)

Re-runs base + trained inference for **Llama-3.2-3B** to produce the per-action JSONs the published hackathon run never had (it shipped `.log` SUMMARY only). The trained LoRA is pulled **directly from HF Hub** (`ddevMhrn/Llama-3.2-3B-Viveka`) — no zip / Kaggle Dataset needed.

**Why:** completes the irreversible-op-accuracy gradient — 1B = 14% (below random) → **3B = ?** → 7B = 88%. With the 3B number, "the skill emerges with capacity" becomes a measured curve, not an assertion.

## Fixes baked in
- Proven install (`pip install -e "."` realigns Kaggle's skewed numpy)
- `torchao` uninstall (peft ≥ 0.14 dispatcher raises on torchao < 0.16, breaks LoRA load)
- Eval base = Unsloth 4-bit mirror (`unsloth/Llama-3.2-3B-Instruct-bnb-4bit`) — meta-llama is gated; inference.py uses plain transformers
- **`CUDA_VISIBLE_DEVICES=0`** on every inference call — prevents the multi-GPU sharding that caused all-abstain on the 1B retrain

## Prereqs
1. **Settings:** Accelerator = `GPU T4 x2`, Internet = `On`, Persistence = `Files only`
2. **Add-ons → Secrets:** `HF_TOKEN` (write scope — pushes results to `ddevMhrn/Llama-3.2-3B-Viveka`)
3. Run cells in order

## Time
~30–45 min (4 passes, per-tier 5). 3B is small.


In [ ]:
# Step 1: GPU check + clone HF Space + tokens ───────────────────────
import os
from kaggle_secrets import UserSecretsClient

EVAL_MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
ADAPTER    = "ddevMhrn/Llama-3.2-3B-Viveka"
REPO_ID    = "ddevMhrn/Llama-3.2-3B-Viveka"
LOG_PREFIX = "llama3b_v2"
RUN_DIR    = "/kaggle/working/runs/llama32_3b_v2_eval"

print(f"Eval base: {EVAL_MODEL}")
print(f"Adapter (from HF): {ADAPTER}")
print(f"Push to: {REPO_ID}")

os.chdir("/"); os.chdir("/kaggle/working")
%cd /kaggle/working
!nvidia-smi | head -20
!mkdir -p /kaggle/working/runs/llama32_3b_v2_eval

!rm -rf /kaggle/working/viveka-env
!git lfs install --skip-repo 2>/dev/null || true
!git clone https://huggingface.co/spaces/ddevMhrn/viveka-env /kaggle/working/viveka-env
%cd /kaggle/working/viveka-env

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
print("\nHF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))


## Step 2: Lean inference install (proven sequence)

`pip install -e "."` first — pulls scikit-learn/scipy from pyproject and realigns Kaggle's base numpy. Then transformers/peft/bnb for model + LoRA loading. torchao removed for the PEFT dispatcher bug.


In [ ]:
# Step 2: Install inference deps ────────────────────────────────────
!pip install -q -e "."
!pip install --upgrade --force-reinstall --no-deps "openenv-core==0.2.2" "fastmcp==3.1.1"
!pip install -q -U "mcp" "uncalled-for"
!pip install -q -U "transformers>=4.40.0" "peft>=0.12" "bitsandbytes" "accelerate>=0.30.0"
!pip install -q -U "huggingface_hub[cli]"
# torchao uninstall — peft >= 0.14 raises on torchao < 0.16 during PeftModel.from_pretrained
!pip uninstall -y torchao 2>&1 | tail -2
print("\n=== versions ===")
!pip show transformers peft bitsandbytes huggingface-hub 2>&1 | grep -E "^(Name|Version)" 


In [ ]:
# Step 3: Verify imports ────────────────────────────────────────────
import importlib, sys
def test(m):
    try: importlib.import_module(m); print(f"\u2705 {m}"); return True
    except Exception as e: print(f"\u274c {m}: {type(e).__name__}: {e}"); return False
ok = True
ok &= test("transformers"); ok &= test("peft"); ok &= test("bitsandbytes")
try:
    from transformers import AutoModelForCausalLM  # noqa: F401
    print("\u2705 transformers.AutoModelForCausalLM")
except Exception as e:
    print(f"\u274c AutoModelForCausalLM: {type(e).__name__}: {e}"); ok = False
try:
    sys.path.insert(0, "/kaggle/working/viveka-env")
    from inference import FrozenQwenPolicy  # noqa: F401
    print("\u2705 inference.FrozenQwenPolicy")
except Exception as e:
    print(f"\u274c inference: {type(e).__name__}: {e}"); ok = False
print(f"\n{'\u2705 ALL CLEAN' if ok else '\u274c FIX BEFORE PROCEEDING'}")


## Inference — base vs trained, T1+T2 then T3+T4

Four passes. Base = frozen `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` (no adapter). Trained = same base + the `ddevMhrn/Llama-3.2-3B-Viveka` LoRA pulled from HF. `CUDA_VISIBLE_DEVICES=0` pins to one GPU so generation doesn't shard and collapse to abstain.


In [ ]:
# Step 4: FROZEN base — T1+T2 ───────────────────────────────────────
out_json = f"{RUN_DIR}/llama3b_v2_base_t12.json"
out_log  = f"{RUN_DIR}/llama3b_v2_base_t12.log"
!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen --model $EVAL_MODEL \
    --tier-mix 1,2 --per-tier 5 \
    --output-json $out_json 2>&1 | tee $out_log


In [ ]:
# Step 5: FROZEN base — T3+T4 ───────────────────────────────────────
out_json = f"{RUN_DIR}/llama3b_v2_base_t34.json"
out_log  = f"{RUN_DIR}/llama3b_v2_base_t34.log"
!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen --model $EVAL_MODEL \
    --tier-mix 3,4 --per-tier 5 \
    --output-json $out_json 2>&1 | tee $out_log


In [ ]:
# Step 6: TRAINED (base + LoRA from HF) — T1+T2 ────────────────────
out_json = f"{RUN_DIR}/llama3b_v2_train_t12.json"
out_log  = f"{RUN_DIR}/llama3b_v2_train_t12.log"
!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen --model $EVAL_MODEL \
    --adapter $ADAPTER \
    --tier-mix 1,2 --per-tier 5 \
    --output-json $out_json 2>&1 | tee $out_log


In [ ]:
# Step 7: TRAINED — T3+T4 (showcase tier) ─────────────────────────
out_json = f"{RUN_DIR}/llama3b_v2_train_t34.json"
out_log  = f"{RUN_DIR}/llama3b_v2_train_t34.log"
!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen --model $EVAL_MODEL \
    --adapter $ADAPTER \
    --tier-mix 3,4 --per-tier 5 \
    --output-json $out_json 2>&1 | tee $out_log

print("\n=== base vs trained summaries (expect VARIED actions, NOT all-abstain) ===")
log_glob = f"{RUN_DIR}/llama3b_v2_*.log"
!grep -A 7 "SUMMARY" $log_glob


## Push the 4 inference files to `ddevMhrn/Llama-3.2-3B-Viveka`

Adds the per-action JSONs/LOGs to the existing 3B repo. After this, pull them locally and run `python eval/capability_report.py --prefix llama3b_v2` for the per-component breakdown + irreversible-op accuracy.


In [ ]:
# Step 8: Push inference results to HF Hub ──────────────────────────
import os, shutil
from pathlib import Path
from huggingface_hub import HfApi, create_repo

RUN_DIR_P = Path(RUN_DIR)
STAGE = Path("/kaggle/working/llama3b_push_stage")
if STAGE.exists(): shutil.rmtree(STAGE)
STAGE.mkdir()
for fname in [
    f"llama3b_v2_base_t12.log",  f"llama3b_v2_base_t12.json",
    f"llama3b_v2_base_t34.log",  f"llama3b_v2_base_t34.json",
    f"llama3b_v2_train_t12.log", f"llama3b_v2_train_t12.json",
    f"llama3b_v2_train_t34.log", f"llama3b_v2_train_t34.json",
]:
    src = RUN_DIR_P / fname
    if src.exists():
        shutil.copy(src, STAGE / fname); print(f"  staged {fname}")
    else:
        print(f"  \u26a0\ufe0f  missing: {fname}")

create_repo(REPO_ID, repo_type="model", exist_ok=True, private=False, token=os.environ["HF_TOKEN"])
HfApi().upload_folder(folder_path=str(STAGE), repo_id=REPO_ID, repo_type="model",
                     token=os.environ["HF_TOKEN"],
                     commit_message="add per-action inference logs (base vs trained) for capacity-gradient analysis")
print(f"\n\u2705 pushed to https://huggingface.co/{REPO_ID}")
